# Task 8: Retrieval-Augmented Generation (RAG) with Llama 3.2\n\n### Objective\nBuild a Retrieval-Augmented Generation (RAG) system that retrieves relevant information from an academic document and uses a locally running Llama 3.2 model through Ollama to answer a user question.\n\n**Pipeline:** Document → Text Extraction → Chunking → Embeddings → Similarity Retrieval → Ollama/Llama 3.2 → Answer\n\nThis notebook is designed to run locally in Jupyter Notebook with Ollama installed.

## 1. Install Required Packages\n\nRun this cell once. For PDF input, `pypdf` is used for text extraction; `sentence-transformers` creates embeddings; `scikit-learn` performs cosine-similarity retrieval; and `ollama` connects to the local model.

In [ ]:
%pip install -q pypdf sentence-transformers scikit-learn numpy ollama

## 2. Check Ollama and Llama 3.2\n\nMake sure Ollama is installed and running before executing this section. If the model is not downloaded yet, run `ollama pull llama3.2` in a terminal.

In [ ]:
import ollama\n\nMODEL_NAME = 'llama3.2'\n\ntry:\n    models = ollama.list()\n    model_names = [m['model'] for m in models.get('models', [])]\n    print('Ollama is running.')\n    print('Available models:', model_names)\n    if not any(name.startswith(MODEL_NAME) for name in model_names):\n        print(f'\nModel {MODEL_NAME} is not available. Run: ollama pull {MODEL_NAME}')\nexcept Exception as e:\n    print('Could not connect to Ollama.')\n    print('Make sure the Ollama application/service is running.')\n    print('Error:', e)

## 3. Imports and Configuration

In [ ]:
from pathlib import Path\nimport numpy as np\nfrom pypdf import PdfReader\nfrom sentence_transformers import SentenceTransformer\nfrom sklearn.metrics.pairwise import cosine_similarity\n\nEMBEDDING_MODEL = 'all-MiniLM-L6-v2'\nTOP_K = 3\nCHUNK_SIZE = 800\nCHUNK_OVERLAP = 150\n

## 4. Provide the Academic Document\n\nPlace an academic PDF in the same folder as this notebook and update `DOCUMENT_PATH`. You can use a syllabus, academic regulations, subject notes, or laboratory manual.\n\nThe fallback text below lets the notebook demonstrate the full RAG flow even before a PDF is added.

In [ ]:
DOCUMENT_PATH = Path('academic_document.pdf')\n\ndef load_document(path: Path) -> str:\n    if path.exists() and path.suffix.lower() == '.pdf':\n        reader = PdfReader(str(path))\n        pages = [(page.extract_text() or '') for page in reader.pages]\n        text = '\\n'.join(pages)\n        if text.strip():\n            return text\n\n    # Small built-in academic sample for testing the pipeline.\n    return '''\nB.Sc. Artificial Intelligence and Machine Learning - Academic Guidelines\n\nStudents are expected to maintain at least 75 percent attendance to be eligible for semester examinations.\nThe assessment pattern consists of continuous internal assessment and an end-semester examination.\nStudents should complete laboratory records and practical work within the deadlines announced by the department.\nThe library provides access to textbooks, reference materials, digital journals, and online learning resources.\nStudents must follow university examination rules, academic integrity requirements, and laboratory safety instructions.\nThe curriculum includes programming, data structures, database systems, operating systems, mathematics, and artificial intelligence subjects.\n'''\n\ndocument_text = load_document(DOCUMENT_PATH)\nprint('Document source:', DOCUMENT_PATH if DOCUMENT_PATH.exists() else 'Built-in sample text')\nprint('Extracted characters:', len(document_text))\nprint('\nPreview:\n', document_text[:1200])

## 5. Split the Document into Chunks\n\nChunking creates smaller passages so that retrieval can return only the parts relevant to the user's question.

In [ ]:
def split_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):\n    text = ' '.join(text.split())\n    if not text:\n        return []\n    chunks = []\n    start = 0\n    step = max(1, chunk_size - overlap)\n    while start < len(text):\n        end = min(start + chunk_size, len(text))\n        chunk = text[start:end].strip()\n        if chunk:\n            chunks.append(chunk)\n        if end >= len(text):\n            break\n        start += step\n    return chunks\n\nchunks = split_text(document_text)\nprint(f'Created {len(chunks)} text chunks.')\nfor i, chunk in enumerate(chunks[:3], start=1):\n    print(f'\n--- Chunk {i} ---\n{chunk}')

## 6. Convert Chunks into Embeddings\n\n`all-MiniLM-L6-v2` is a lightweight sentence-transformer model suitable for local semantic search.

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)\nchunk_embeddings = embedding_model.encode(\n    chunks,\n    convert_to_numpy=True,\n    normalize_embeddings=True,\n)\n\nprint('Embedding model:', EMBEDDING_MODEL)\nprint('Embedding matrix shape:', chunk_embeddings.shape)

## 7. Retrieve the Most Relevant Context

In [ ]:
def retrieve_context(question: str, top_k: int = TOP_K):\n    question_embedding = embedding_model.encode(\n        [question],\n        convert_to_numpy=True,\n        normalize_embeddings=True,\n    )\n    scores = cosine_similarity(question_embedding, chunk_embeddings)[0]\n    top_indices = np.argsort(scores)[::-1][:top_k]\n    results = [(int(i), float(scores[i]), chunks[i]) for i in top_indices]\n    return results\n

## 8. Ask a Question and Generate the Answer with Local Llama 3.2\n\nThe retrieved chunks are placed into the prompt as context. The model is instructed to answer from that context and not invent facts outside it.

In [ ]:
question = input('Enter your academic question: ').strip()\nif not question:\n    question = 'What is the minimum attendance required for semester examinations?'\n\nretrieved = retrieve_context(question, TOP_K)\ncontext = '\\n\\n'.join(\n    f'[Retrieved Chunk {rank} | similarity={score:.3f}]\\n{text}'\n    for rank, (_, score, text) in enumerate(retrieved, start=1)\n)\n\nprompt = f'''You are an academic document assistant.\nAnswer the user's question using only the retrieved context below.\nIf the answer is not present in the context, say that the information was not found in the document.\nDo not invent rules, dates, numbers, or policies.\n\nRetrieved context:\n{context}\n\nUser question: {question}\n\nAnswer clearly and concisely.'''\n\ntry:\n    response = ollama.chat(\n        model=MODEL_NAME,\n        messages=[{'role': 'user', 'content': prompt}],\n    )\n    answer = response['message']['content']\nexcept Exception as e:\n    answer = f'Unable to generate the local Llama 3.2 response. Error: {e}'\n\nprint('\\n' + '=' * 70)\nprint('USER QUESTION')\nprint('=' * 70)\nprint(question)\n\nprint('\\n' + '=' * 70)\nprint('RETRIEVED RELEVANT CONTEXT')\nprint('=' * 70)\nfor rank, (idx, score, text) in enumerate(retrieved, start=1):\n    print(f'\\nChunk {rank} (original index: {idx}, similarity: {score:.3f})')\n    print(text)\n\nprint('\\n' + '=' * 70)\nprint('FINAL RESPONSE FROM LLAMA 3.2')\nprint('=' * 70)\nprint(answer)

## 9. RAG Workflow Summary\n\n1. **Document ingestion:** The academic PDF is read with `pypdf`.\n2. **Chunking:** The document is split into overlapping text chunks.\n3. **Embedding:** Each chunk is converted into a semantic vector using Sentence Transformers.\n4. **Retrieval:** The question is embedded and compared with chunk embeddings using cosine similarity.\n5. **Generation:** The top relevant chunks and the question are sent to local **Llama 3.2** through Ollama.\n6. **Output:** The notebook displays the user's question, retrieved context, and final generated answer.\n\n### Expected Result\nThe notebook demonstrates a complete local RAG pipeline without sending the academic document to a hosted LLM API.

## 10. Requirements\n\n- Python 3.9+\n- Jupyter Notebook\n- Ollama installed and running\n- Local model: `llama3.2`\n- An academic PDF such as a syllabus, regulations, notes, or lab manual\n\n### Ollama setup\nRun in a terminal:\n```bash\nollama pull llama3.2\n```\n\nThen start Ollama and execute the notebook cells in order.